In [2]:
import sys
import os
sys.path.append(os.path.abspath('../../'))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from functions.preproc import *
from functions.ensembles import *

from sklearn import set_config
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import make_scorer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import StratifiedKFold
from sksurv.metrics import integrated_brier_score
from sksurv.metrics import concordance_index_ipcw
from sksurv.nonparametric import kaplan_meier_estimator
from sklearn.inspection import permutation_importance

from sksurv.ensemble import RandomSurvivalForest
from sksurv.metrics import concordance_index_censored
from sksurv.util import Surv

import time
set_config(display="text")  # displays text representation of estimators

In [3]:
dataset = pd.read_csv("../data/data_final.csv")

diagnosis_variables = []

with open("../data/diagnosis_final.txt", "r") as file:
    for line in file:
        diagnosis_variables.append(line.strip())

treatment_variables = []
with open("../data/treatment_final.txt", "r") as file:
    for line in file:
        treatment_variables.append(line.strip())

survival_dataset = dataset.drop('Unnamed: 0', axis=1)

feature_columns = treatment_variables + diagnosis_variables

In [4]:
# Find duplicated patients
mask = (survival_dataset.duplicated(subset='PERSON_ID', keep=False)) #& (survival_dataset['Regional LN recurrence'] == 1)

# For each duplicated PERSON_ID with recurrence, keep only the row with the lowest survival days
to_remove = (
    survival_dataset[mask]
    .sort_values(['PERSON_ID', 'Survival days'], ascending=[True, True])
    .groupby('PERSON_ID')
    .head(-1)
    .index
)

survival_dataset = survival_dataset.drop(to_remove)

In [5]:
# Prepare data
data_x = survival_dataset[feature_columns]
data_y = Surv.from_arrays(
    event=survival_dataset['Regional LN recurrence'],
    time=survival_dataset['Survival days']
)

In [6]:
sgkf = StratifiedKFold(n_splits=5, shuffle=True, random_state=12)
cv = sgkf.split(X=data_x, y=survival_dataset['Regional LN recurrence'])


train_index, test_index = next(cv)
x_train = data_x.iloc[train_index]
y_train = data_y[train_index]

x_test = data_x.iloc[test_index]
y_test = data_y[test_index]

subsampling = survival_dataset.iloc[train_index].copy()

In [6]:
for train_idx, val_idx in cv:
    print("Train events:", survival_dataset.iloc[train_idx]["Regional LN recurrence"].sum())
    print("Val events:", survival_dataset.iloc[val_idx]["Regional LN recurrence"].sum())

Train events: 27.0
Val events: 6.0
Train events: 26.0
Val events: 7.0
Train events: 26.0
Val events: 7.0
Train events: 26.0
Val events: 7.0


In [7]:
'''data_x = survival_dataset[feature_columns]
data_y = Surv.from_arrays(
    event=survival_dataset['Regional LN recurrence'],
    time=survival_dataset['Survival days']
)
groups = survival_dataset['PERSON_ID']
event_indicator = survival_dataset['Regional LN recurrence']

time = data_y["time"]
valid_mask = time < 0

data_x = data_x[valid_mask]
data_y = data_y[valid_mask]
groups = groups[valid_mask]
event_indicator = event_indicator[valid_mask]'''

'data_x = survival_dataset[feature_columns]\ndata_y = Surv.from_arrays(\n    event=survival_dataset[\'Regional LN recurrence\'],\n    time=survival_dataset[\'Survival days\']\n)\ngroups = survival_dataset[\'PERSON_ID\']\nevent_indicator = survival_dataset[\'Regional LN recurrence\']\n\ntime = data_y["time"]\nvalid_mask = time < 0\n\ndata_x = data_x[valid_mask]\ndata_y = data_y[valid_mask]\ngroups = groups[valid_mask]\nevent_indicator = event_indicator[valid_mask]'

In [8]:
'''pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", FastSurvivalSVM(random_state=12))
])'''

param_grid = {
    'alpha': [0.01, 0.1, 1.0, 10.0, 100.0],
    'rank_ratio': [0.0, 0.5, 1.0],
    'tol': [1e-3, 1e-4],
    'max_iter': [5000, 10000],
    'n_controls_per_case': [1, 2, 4, 10, 15]
}

# Best parameters: {'svm__alpha': 0.01, 'svm__max_iter': 5000, 'svm__rank_ratio': 1.0, 'svm__tol': 0.0001}


In [9]:
sgkf_1 = StratifiedKFold(n_splits=4, shuffle=True, random_state=12)
cv = sgkf_1.split(X=x_train, y=y_train['event'])

In [10]:
results_df, best_params, summary = generic_grid_search(
    FastSurvivalSVM,
    x_train,
    y_train,
    cv,
    param_grid,
    metrics=['c-index', 'c-index-ipcw'], #it can't measure int-brier-score since it doesnt prodce surv curves
    model_name='SVM',
    scale_data=True,
    scaler_type='standard',
    verbose=True,
    random_state=12,
    n_jobs=-1,
    dataset=subsampling,
    feature_columns=feature_columns
)


SVM GRID SEARCH (Manual Implementation)
Model: FastSurvivalSVM
Scaling: Yes (standard) - INSIDE each fold
Subsampling: No
Metrics: ['c-index', 'c-index-ipcw']
Parameter Grid:
  - alpha: [0.01, 0.1, 1.0, 10.0, 100.0]
  - rank_ratio: [0.0, 0.5, 1.0]
  - tol: [0.001, 0.0001]
  - max_iter: [5000, 10000]
  - n_controls_per_case: [1, 2, 4, 10, 15]
Total combinations: 300

[1/300] Testing params: {'alpha': 0.01, 'rank_ratio': 0.0, 'tol': 0.001, 'max_iter': 5000}, n_controls_per_case=1
  c-index: 0.2265 ± 0.0893
  c-index-ipcw: 0.3229 ± 0.0988
[2/300] Testing params: {'alpha': 0.01, 'rank_ratio': 0.0, 'tol': 0.001, 'max_iter': 5000}, n_controls_per_case=2
  Elapsed: 0.0min, Est. remaining: 1.7min
  c-index: 0.2796 ± 0.1285
  c-index-ipcw: 0.3405 ± 0.1224
[3/300] Testing params: {'alpha': 0.01, 'rank_ratio': 0.0, 'tol': 0.001, 'max_iter': 5000}, n_controls_per_case=4
  Elapsed: 0.0min, Est. remaining: 1.6min
  c-index: 0.3259 ± 0.0744
  c-index-ipcw: 0.4264 ± 0.0792
[4/300] Testing params: {'a

In [11]:
results_df.to_csv("../results/subsampling/ssvm_subsampling_grid_search_results.csv", index=False)

In [16]:
results_df

,alpha,rank_ratio,tol,max_iter,n_controls_per_case,c-index_mean,c-index_std,c-index-ipcw_mean,c-index-ipcw_std,c-index_fold_1,c-index_fold_2,c-index_fold_3,c-index_fold_4,c-index-ipcw_fold_1,c-index-ipcw_fold_2,c-index-ipcw_fold_3,c-index-ipcw_fold_4,c-index_has_error,c-index-ipcw_has_error
0,0.01,0.0,0.0010,5000,1,0.226490,0.089326,0.322873,0.098826,0.137996,0.148027,0.354645,0.265289,0.393521,0.152511,0.366392,0.379069,False,False
1,0.01,0.0,0.0010,5000,2,0.279645,0.128511,0.340454,0.122448,0.117832,0.217443,0.465534,0.317769,0.240609,0.233246,0.536381,0.351581,False,False
2,0.01,0.0,0.0010,5000,4,0.325896,0.074421,0.426384,0.079213,0.264020,0.350934,0.436563,0.252066,0.500469,0.348376,0.510555,0.346135,False,False
3,0.01,0.0,0.0010,5000,10,0.382557,0.047900,0.451965,0.052075,0.322621,0.419460,0.438561,0.349587,0.381805,0.425980,0.517754,0.482322,False,False
4,0.01,0.0,0.0010,5000,15,0.432528,0.076957,0.507921,0.032364,0.484562,0.497775,0.444056,0.303719,0.547334,0.516000,0.511040,0.457312,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
295,100.00,1.0,0.0001,10000,1,0.631955,0.085572,0.615686,0.095583,0.500315,0.739247,0.635365,0.652893,0.483847,0.753750,0.618973,0.606173,False,False
296,100.00,1.0,0.0001,10000,2,0.719489,0.090732,0.680545,0.103092,0.804663,0.810739,0.601399,0.661157,0.722129,0.800841,0.518336,0.680873,False,False
297,100.00,1.0,0.0001,10000,4,0.640327,0.132393,0.600164,0.169392,0.515438,0.863245,0.588412,0.594215,0.407515,0.862176,0.508014,0.622952,False,False
298,100.00,1.0,0.0001,10000,10,0.686381,0.131131,0.637142,0.165826,0.538122,0.888164,0.611389,0.707851,0.418489,0.858371,0.554495,0.717213,False,False


# Train and Test Final Model with Best Hyperparameters

In [7]:
# Load best hyperparameters from corrected_sub_results
best_results = pd.read_csv("../results/corrected_sub_results/ssvm_subsampling_grid_search_results.csv")
# Select row with highest c-index-ipcw_mean
best_params = best_results.sort_values('c-index-ipcw_mean', ascending=False).iloc[0]

print("Best hyperparameters:")
print(f"  alpha: {best_params['alpha']}")
print(f"  rank_ratio: {best_params['rank_ratio']}")
print(f"  tol: {best_params['tol']}")
print(f"  max_iter: {int(best_params['max_iter'])}")
print(f"  n_controls_per_case: {int(best_params['n_controls_per_case'])}")
print(f"  c-index-ipcw_mean (CV): {best_params['c-index-ipcw_mean']:.4f}")
print()

Best hyperparameters:
  alpha: 0.01
  rank_ratio: 1.0
  tol: 0.001
  max_iter: 5000
  n_controls_per_case: 2
  c-index-ipcw_mean (CV): 0.7327



In [8]:
# Prepare training fold dataframe (subsampling variable was created earlier)
train_fold = subsampling.copy()

# Apply case-control subsampling
x_train_sub, y_train_sub, sub_dataset = sample_case_control(
    train_fold=train_fold,
    event_col='Regional LN recurrence',
    time_col='Survival days',
    n_controls_per_case=int(best_params['n_controls_per_case']),
    feature_columns=feature_columns,
    random_state=12
)

print(f"Original training samples: {len(x_train)}")
print(f"Subsampled training samples: {len(x_train_sub)}")
print(f"Test samples: {len(x_test)}")

Original training samples: 2376
Subsampled training samples: 81
Test samples: 594


In [9]:
# Train final model with best hyperparameters (SVM needs scaling)
scaler = StandardScaler()
x_train_sub_scaled = scaler.fit_transform(x_train_sub)

final_model = FastSurvivalSVM(
    alpha=best_params['alpha'],
    rank_ratio=best_params['rank_ratio'],
    tol=best_params['tol'],
    max_iter=int(best_params['max_iter']),
    random_state=12
)

final_model.fit(x_train_sub_scaled, y_train_sub)
print("Model trained successfully!")

Model trained successfully!


In [10]:
# Test the model on test set
x_test_scaled = scaler.transform(x_test)

# Calculate c-index
c_index = final_model.score(x_test_scaled, y_test)

# Calculate c-index IPCW
tau = y_train_sub["time"].max()
c_index_ipcw = concordance_index_ipcw(y_train_sub, y_test, final_model.predict(x_test_scaled), tau=tau)[0]

print("\\nTest Set Performance:")
print(f"  C-index: {c_index:.4f}")
print(f"  C-index IPCW: {c_index_ipcw:.4f}")
print("  Note: FastSurvivalSVM does not produce survival curves, so Brier score cannot be calculated.")

\nTest Set Performance:
  C-index: 0.7661
  C-index IPCW: 0.7503
  Note: FastSurvivalSVM does not produce survival curves, so Brier score cannot be calculated.


# Feature Importance for SHAP Analysis

In [13]:
# Extract feature importance from SSVM model
# For SVM models, feature importance is based on the absolute value of coefficients
import pandas as pd
import numpy as np

# IMPORTANT: Use the actual feature names from the training data
# The model was trained on x_train_sub_scaled, which came from x_train_sub
actual_features = x_train_sub.columns.tolist()

# Get coefficients from the final SSVM model
# FastSurvivalSVM stores coefficients in coef_ attribute
coefficients = final_model.coef_

# Handle different shapes
if coefficients.ndim > 1:
    coefficients = coefficients.flatten()

# Ensure coefficient length matches the actual features used
if len(coefficients) != len(actual_features):
    print(f"Warning: Coefficient length ({len(coefficients)}) doesn't match feature length ({len(actual_features)})")
    print(f"Using minimum length for mapping")
    min_len = min(len(coefficients), len(actual_features))
    coefficients = coefficients[:min_len]
    actual_features = actual_features[:min_len]

coef_df = pd.DataFrame({
    'Feature': actual_features,
    'Coefficient': coefficients,
    'Abs_Coefficient': np.abs(coefficients)
})

# Sort by absolute value of coefficient (most important first)
coef_df = coef_df.sort_values('Abs_Coefficient', ascending=False)

# Display top 20 most important features
print("Top 20 Most Important Features for SSVM Model:")
print("="*60)
print(coef_df.head(20).to_string(index=False))
print(f"\nTotal features: {len(coef_df)}")
print(f"Total non-zero coefficients: {(coef_df['Coefficient'] != 0).sum()}")

# Store for SHAP analysis
top_features_ssvm = coef_df.head(20)['Feature'].tolist()
print(f"\n\nTop features stored in 'top_features_ssvm' variable")
print(f"All coefficients stored in 'coef_df' variable")

Top 20 Most Important Features for SSVM Model:
                                                  Feature  Coefficient  Abs_Coefficient
                                       Istotype at CB_1.0     0.285955         0.285955
                      Oestrogen receptor status at CB_1.0     0.273786         0.273786
               Treatment in association with chemotherapy    -0.225116         0.225116
                                       Istotype at CB_4.0    -0.220999         0.220999
                          Adjuvant therapy with bio drugs     0.214042         0.214042
Her2 overexpression (with immunohystochemistry) at CB_1.0    -0.212193         0.212193
                               Radiotherapy on chest wall    -0.175549         0.175549
                      Oestrogen receptor status at CB_0.0    -0.160277         0.160277
                              Endocrine therapy performed    -0.159271         0.159271
                                  Boost dose administered     0.152807   

In [15]:
# Export feature importance to CSV
coef_df.to_csv("../results/features/ssvm_feature_importance.csv", index=False)
print("Feature importance exported to: ../results/features/ssvm_feature_importance.csv")

Feature importance exported to: ../results/features/ssvm_feature_importance.csv
